In [ ]:
from kafka.admin import KafkaAdminClient, NewTopic

admin_client = KafkaAdminClient(
    bootstrap_servers="localhost:9092",
    client_id="test-admin"
)

topic_list = []

topic_list.append(
    NewTopic(
        name="Sensors-IoT",
        num_partitions=1,
        replication_factor=1
    )
)

admin_client.create_topics(new_topics=topic_list, validate_only=False)

print("Topic creado")

TopicAlreadyExistsError: [Error 36] TopicAlreadyExistsError: Request 'CreateTopicsRequest_v3(create_topic_requests=[(topic='Sensors-IoT', num_partitions=1, replication_factor=1, replica_assignment=[], configs=[])], timeout=30000, validate_only=False)' failed with response 'CreateTopicsResponse_v3(throttle_time_ms=0, topic_errors=[(topic='Sensors-IoT', error_code=36, error_message="Topic 'Sensors-IoT' already exists.")])'.

In [8]:
from kafka import KafkaProducer
import json
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)


In [ ]:
import time
import random
from datetime import datetime
import pandas as pd

random.seed(42)

locations = [
    {"sensor_id": "BCN_001", "ubicacion": "Ronda Litoral",     "lat": 41.380, "lon": 2.180, "base_flow": 50, "base_speed": 45},
    {"sensor_id": "BCN_002", "ubicacion": "Ronda de Dalt",     "lat": 41.441, "lon": 2.134, "base_flow": 48, "base_speed": 50},
    {"sensor_id": "BCN_003", "ubicacion": "Diagonal",          "lat": 41.393, "lon": 2.162, "base_flow": 42, "base_speed": 28},
    {"sensor_id": "BCN_004", "ubicacion": "Passeig de Gracia", "lat": 41.392, "lon": 2.165, "base_flow": 30, "base_speed": 22},
    {"sensor_id": "BCN_005", "ubicacion": "Gran Via",          "lat": 41.387, "lon": 2.149, "base_flow": 44, "base_speed": 26},
    {"sensor_id": "BCN_006", "ubicacion": "Meridiana",         "lat": 41.429, "lon": 2.183, "base_flow": 46, "base_speed": 30},
    {"sensor_id": "BCN_007", "ubicacion": "Placa Catalunya",   "lat": 41.387, "lon": 2.170, "base_flow": 26, "base_speed": 18},
    {"sensor_id": "BCN_008", "ubicacion": "Placa Espanya",     "lat": 41.375, "lon": 2.149, "base_flow": 34, "base_speed": 20},
    {"sensor_id": "BCN_009", "ubicacion": "Via Laietana",      "lat": 41.385, "lon": 2.177, "base_flow": 24, "base_speed": 18},
    {"sensor_id": "BCN_010", "ubicacion": "Sants",             "lat": 41.379, "lon": 2.140, "base_flow": 28, "base_speed": 20},
]

def hour_multiplier(ts: datetime) -> float:
    hour = ts.hour
    weekday = ts.weekday()

    if weekday < 5:
        if 7 <= hour <= 9:
            return 1.7
        elif 13 <= hour <= 15:
            return 1.2
        elif 17 <= hour <= 20:
            return 1.8
        elif 0 <= hour <= 5:
            return 0.35
        else:
            return 1.0
    else:
        if 11 <= hour <= 14:
            return 1.15
        elif 18 <= hour <= 22:
            return 1.25
        elif 0 <= hour <= 6:
            return 0.40
        else:
            return 0.80

def congestion_label(occupancy_pct: float) -> str:
    if occupancy_pct < 30:
        return "low"
    elif occupancy_pct < 55:
        return "medium"
    elif occupancy_pct < 80:
        return "high"
    return "very_high"

def simulate_sensor_row(location: dict, ts: datetime) -> dict:
    hm = hour_multiplier(ts)

    incident = 1 if random.random() < 0.03 else 0

    vehicles = location["base_flow"] * hm
    vehicles *= random.uniform(0.85, 1.15)

    if incident:
        vehicles *= 1.20

    vehicles_per_minute = max(0, round(vehicles))

    speed = location["base_speed"] * random.uniform(0.90, 1.10)
    speed -= max(0, vehicles_per_minute - location["base_flow"]) * 0.5

    if incident:
        speed -= random.uniform(8, 15)

    avg_speed_kmh = max(5, round(speed, 1))

    occupancy_pct = min(
        100,
        max(
            5,
            round(20 + vehicles_per_minute * 1.3 - avg_speed_kmh * 0.4 + random.uniform(-4, 4), 1)
        )
    )

    return {
        "sensor_id": location["sensor_id"],
        "timestamp": ts.isoformat(),
        "ubicacion": location["ubicacion"],
        "lat": location["lat"],
        "lon": location["lon"],
        "vehicles_per_minute": vehicles_per_minute,
        "avg_speed_kmh": avg_speed_kmh,
        "occupancy_pct": occupancy_pct,
        "congestion_level": congestion_label(occupancy_pct),
        "incident": incident
    }

def generate_realtime_batch() -> pd.DataFrame:
    ts = datetime.now()
    rows = [simulate_sensor_row(loc, ts) for loc in locations]
    return pd.DataFrame(rows)

while True:
    df_batch = generate_realtime_batch()
    producer.send("Sensors-IoT",df_batch.to_dict())
    producer.flush()
    print("-" * 80)
    time.sleep(5)  # genera una tanda cada 5 segundos